In [ ]:
###
# 1. 環境のセットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/cnn-hands-on')
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

os.chdir(ROOT_PATH)

# 日本語フォント対応
!pip install -q japanize-matplotlib
import japanize_matplotlib

print(f"✅ 環境セットアップ完了！現在のディレクトリ: {Path.cwd()}")

# 1. プーリング層とは

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from PIL import ImageOps

# 画像を読み込む
img_path = ROOT_PATH / "data" / "cats_vs_dogs" / "Cat" / "00000.jpg"
img = Image.open(img_path)

# グレースケールに変換
img_small = transforms.Resize((28, 28))(img)
img_gray = ImageOps.grayscale(img_small)
img_array = np.array(img_gray)

print(f"入力画像サイズ: {img_array.shape}")
plt.imshow(img_array, cmap='gray')
plt.title('Input Image (28x28)')
plt.show()

---

In [ ]:
# MaxPoolingの動作を視覚化
sample = np.array([
    [1, 3, 2, 1],
    [4, 2, 3, 1],
    [2, 4, 1, 2],
    [3, 1, 2, 3]
])

print("入力 (4x4):")
print(sample)
print()
print("MaxPooling (2x2) の結果:")
print("左上2x2: max([1,3,4,2]) =", np.max(sample[0:2, 0:2]))
print("右上2x2: max([2,1,3,1]) =", np.max(sample[0:2, 2:4]))
print("左下2x2: max([2,4,3,1]) =", np.max(sample[2:4, 0:2]))
print("右下2x2: max([1,2,2,3]) =", np.max(sample[2:4, 2:4]))

# 2. プーリングの実装

In [ ]:
###
# 演習1: MaxPoolingを手動で実装【解答】
###

def max_pool2d(img, pool_size=2):
    # 入力画像の高さと幅を取得
    h, w = img.shape[:2]
    
    # 出力サイズを計算（入力サイズ // pool_size）
    out_h = h // pool_size
    out_w = w // pool_size
    
    # 出力配列を初期化
    out = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            # pool_size × pool_size の領域を切り出す
            region = img[
                i*pool_size:(i+1)*pool_size,
                j*pool_size:(j+1)*pool_size
            ]
            # 最大値を取得
            out[i, j] = np.max(region)
    return out

In [ ]:
# テスト
test_img = np.array([
    [1, 3, 2, 1],
    [4, 2, 3, 1],
    [2, 4, 1, 2],
    [3, 1, 2, 3]
])

result = max_pool2d(test_img, pool_size=2)
print("入力:")
print(test_img)
print("\nMaxPooling結果:")
print(result)
print("\n期待される出力: [[4, 3], [4, 3]]")

---

In [ ]:
###
# 発展演習: AveragePoolingも実装【解答】
###

def avg_pool2d(img, pool_size=2):
    h, w = img.shape[:2]
    out_h = h // pool_size
    out_w = w // pool_size
    out = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = img[
                i*pool_size:(i+1)*pool_size,
                j*pool_size:(j+1)*pool_size
            ]
            # np.mean() で平均値を取得
            out[i, j] = np.mean(region)
    return out

In [ ]:
# テスト
result_avg = avg_pool2d(test_img, pool_size=2)
print("入力:")
print(test_img)
print("\nAveragePooling結果:")
print(result_avg)
print("\n期待される出力: [[2.5, 1.75], [2.5, 2.0]]")

---

In [ ]:
# 実際の画像に適用
pooled_max = max_pool2d(img_array, pool_size=2)
pooled_avg = avg_pool2d(img_array, pool_size=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_array, cmap='gray')
axes[0].set_title(f'Input ({img_array.shape[0]}x{img_array.shape[1]})')

axes[1].imshow(pooled_max, cmap='gray')
axes[1].set_title(f'MaxPooling ({pooled_max.shape[0]}x{pooled_max.shape[1]})')

axes[2].imshow(pooled_avg, cmap='gray')
axes[2].set_title(f'AvgPooling ({pooled_avg.shape[0]}x{pooled_avg.shape[1]})')

plt.tight_layout()
plt.show()

---

In [ ]:
###
# PyTorchでのプーリング
###

import torch
import torch.nn as nn

# MaxPooling層を定義
pool = nn.MaxPool2d(
    kernel_size=2,  # 領域サイズ
    stride=2        # 移動幅
)

# 推論 (B, C, H, W) 形式
x = torch.randn(1, 1, 28, 28)
output = pool(x)

print(f"入力サイズ: {x.shape}")
print(f"出力サイズ: {output.shape}")

# 3. 活性化関数とは

In [ ]:
# 代表的な活性化関数の可視化
x = np.linspace(-5, 5, 100)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))

# Tanh
tanh = np.tanh(x)

# ReLU
relu = np.maximum(0, x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, sigmoid, color='orange', linewidth=2)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_title('Sigmoid (0~1)')
axes[0].set_ylim(-0.5, 1.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, tanh, color='purple', linewidth=2)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_title('Tanh (-1~1)')
axes[1].set_ylim(-1.5, 1.5)
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, relu, color='blue', linewidth=2)
axes[2].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[2].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[2].set_title('ReLU（主流）')
axes[2].set_ylim(-1, 6)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. 活性化関数の実装

In [ ]:
###
# 演習2: ReLUを手動で実装【解答】
###

def relu(x):
    # np.maximum(a, b): 要素ごとに大きい方を返す
    return np.maximum(0, x)

In [ ]:
# テスト
test_x = np.array([-2, -1, 0, 1, 2])
result = relu(test_x)
print(f"入力: {test_x}")
print(f"ReLU結果: {result}")
print(f"期待される出力: [0, 0, 0, 1, 2]")

---

In [ ]:
###
# 別の実装方法（条件付き代入）【解答】
###

def relu_v2(x):
    out = np.copy(x)
    # 0以下の値を0に置き換える
    out[out <= 0] = 0
    return out

In [ ]:
# 特徴マップに適用
feature_map = np.array([
    [-1, 2, -3],
    [4, -5, 6]
])

print("入力特徴マップ:")
print(feature_map)
print("\nReLU後:")
print(relu(feature_map))
print("\n期待される出力: [[0, 2, 0], [4, 0, 6]]")

---

In [ ]:
###
# PyTorchでの活性化関数
###

import torch
import torch.nn as nn
import torch.nn.functional as F

# ReLU層を定義
relu_layer = nn.ReLU()

# 推論
x = torch.tensor([-2., -1., 0., 1., 2.])
output = relu_layer(x)
print(f"nn.ReLU: {output}")

# 関数としても使える
output2 = F.relu(x)
print(f"F.relu: {output2}")

# 5. Conv → ReLU → Pool パイプライン

In [ ]:
###
# 発展演習: CNNの1ブロックを組む
###

import torch
import torch.nn as nn

# 各層を定義
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# 入力（1枚、1ch、28×28）
x = torch.randn(1, 1, 28, 28)
print(f"入力: {x.shape}")

# 順番に適用
x = conv(x)
print(f"Conv後: {x.shape}")

x = relu(x)
print(f"ReLU後: {x.shape}")

x = pool(x)
print(f"Pool後: {x.shape}")

In [ ]:
###
# 複数ブロックを重ねる
###

# 28x28 → 14x14 → 7x7
conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

x = torch.randn(1, 1, 28, 28)
print(f"入力: {x.shape}")

# Block 1
x = conv1(x)
x = relu(x)
x = pool(x)
print(f"Block1後: {x.shape}")

# Block 2
x = conv2(x)
x = relu(x)
x = pool(x)
print(f"Block2後: {x.shape}")

# 6. まとめ

- **プーリング層**: 特徴マップを縮小し、位置ズレに強くする。MaxPooling（最大値）が主流、学習パラメータなし
- **活性化関数**: 非線形性を加え、深いネットワークに意味を持たせる。ReLU（負→0、正→そのまま）が主流
- **CNNの基本ブロック**: `Conv → ReLU → Pool` の組み合わせが1セット